# **HANDS - ON: FUNDAMENTOS DE LLMS CON MODELOS LLAMA**


Una vez vista la masterclass ***Fundamentos de LLMs y Arquitectura de Llama***, se proporciona el siguiente ***Colab*** para construir, en vivo, la primera llamada a Llama y medir su comportamiento.

Usamos **Groq** para tener acceso a inferencia de Llama con GPU gratuita, sin necesidad de infraestructura propia.

## **CONFIGURACIÓN DEL ENTORNO**

Antes de llamar a Llama necesitamos activar la GPU gratuita de Colab y conectar nuestra cuenta de **Groq**. La librería `groq` es el cliente oficial en Python para hacer llamadas a la API.

### **COLAB SECRETS**

Para no exponer tu ***API key*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la ***API key*** para guardarla dentro de una variable y usarla dentro del notebook.

In [1]:
# ============================================================
# PASO 1. INSTALAR GROQ Y CONECTAR LA API KEY DE FORMA SEGURA
# ============================================================
#
# !pip install groq -q
# Instala el cliente de Python que nos permite comunicarnos con
# la API de Groq. El parámetro -q hace que la instalación muestre
# menos mensajes en pantalla.

!pip install groq -q

# Importamos la clase Groq, que será nuestro cliente para hacer
# solicitudes al modelo.
from groq import Groq

# userdata permite leer variables guardadas en "Colab Secrets".
# Así evitamos escribir la API key directamente en el notebook.
from google.colab import userdata

# IMPORTANTE:
# Antes de ejecutar esta celda:
# 1. Abre el ícono de llave (Secrets) en la barra lateral de Colab.
# 2. Crea un secreto llamado exactamente: GROQ_API_KEY
# 3. Pega ahí tu API key de Groq.
# 4. Activa el acceso del notebook al secreto.
#
# userdata.get(...) recupera el secreto y Groq(...) crea el cliente.
client = Groq(api_key=userdata.get("GROQ_API_KEY"))

print("Cliente de Groq inicializado correctamente.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 2.1 MB/s eta 0:00:00
Cliente de Groq inicializado correctamente.


## **DE PALABRAS A TOKENS**

Antes de que Llama genere una sola palabra, tu prompt se divide en **tokens**. Vamos a definir un prompt de ejemplo que después enviaremos a Llama.

In [2]:
# ============================================================
# PASO 2. DEFINIR EL PROMPT
# ============================================================
#
# Un prompt es la instrucción o pregunta que enviaremos al modelo.
# En este Hands On usaremos una pregunta sencilla para observar:
# - la respuesta generada,
# - los tokens consumidos,
# - y el tiempo de respuesta.

prompt = "¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?"

# Puedes probar después con otra pregunta, por ejemplo:
# prompt = "Explica en un párrafo qué hace un router doméstico"

print("Prompt que enviaremos al modelo:")
print(prompt)


Prompt que enviaremos al modelo:
¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?


In [3]:
#comprobar modelos de la cuenta
models = client.models.list()

for model in models.data:
    print(model.id)

meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-120b
openai/gpt-oss-safeguard-20b
canopylabs/orpheus-v1-english
whisper-large-v3-turbo
canopylabs/orpheus-arabic-saudi
allam-2-7b
meta-llama/llama-prompt-guard-2-22m
openai/gpt-oss-20b
whisper-large-v3
qwen/qwen3.8-27b


### **PRIMERA LLAMADA A LLAMA (ZERO-SHOT)**

Un LLM predice el siguiente token más probable, **no busca la respuesta en una base de datos**. Vamos a enviar el prompt a Llama en modo zero-shot (sin ejemplos previos) y ver la respuesta generada.

In [4]:
# ============================================================
# PASO 3. HACER LA PRIMERA LLAMADA A LLAMA (ZERO-SHOT)
# ============================================================
#
# Zero-shot significa que pedimos una tarea directamente,
# sin proporcionar ejemplos previos de cómo debe responder.
#
# client.chat.completions.create(...) envía la solicitud a Groq.
# model indica qué modelo Llama queremos utilizar.
# messages contiene la conversación que recibirá el modelo.
#
# Aquí solo enviamos un mensaje con role="user".

response = client.chat.completions.create(
    model="openai/gpt-oss-20b", #<--- usamos penai
    messages=[
        {"role": "user", "content": prompt}
    ]
)



# La respuesta completa contiene bastante información.
# El texto generado está dentro de:
# response.choices[0].message.content
print(response.choices[0].message.content)


**RAM (Random Access Memory) vs. Almacenamiento (Disco Duro / SSD)**  

| Característica | RAM | Almacenamiento (HDD / SSD) |
|----------------|-----|---------------------------|
| **Volatilidad** | Volátil: se borra cuando la computadora se apaga. | No volátil: conserva los datos incluso sin energía. |
| **Propósito** | Guarda temporalmente los datos y programas que la CPU necesita usar inmediatamente. | Guarda permanentemente el sistema operativo, programas, archivos de usuario y datos. |
| **Velocidad** | Extremadamente rápido (microsegundos). | Más lento, aunque un SSD ya es mucho más rápido que un HDD tradicional (milisegundos). |
| **Acceso** | Acceso aleatorio y casi instantáneo a cualquier celda. | Acceso aleatorio, pero con latencia mayor y, en HDD, movimiento mecánico. |
| **Capacidad típica** | Desde 2 GB hasta 64 GB (o más en servidores). | Desde 128 GB en SSDs económicos hasta varios terabytes en discos duros de escritorio. |
| **Costo por GB** | Más caro por GB que el alm

In [5]:
# ============================================================
# PASO 4. INSPECCIONAR LA RESPUESTA COMPLETA DE LA API
# ============================================================
#
# Hasta ahora solo mostramos el texto generado.
# Sin embargo, la API devuelve además metadatos, por ejemplo:
# - modelo utilizado,
# - información de uso,
# - motivo de finalización,
# - tokens consumidos, etc.
#
# model_dump_json(indent=2) convierte el objeto completo
# a un JSON legible para poder inspeccionarlo.

print(response.model_dump_json(indent=2))


{
  "id": "chatcmpl-e34d1770-7cee-48f7-b967-9af1dea88546",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "**RAM (Random Access Memory) vs. Almacenamiento (Disco Duro / SSD)**  \n\n| Característica | RAM | Almacenamiento (HDD / SSD) |\n|----------------|-----|---------------------------|\n| **Volatilidad** | Volátil: se borra cuando la computadora se apaga. | No volátil: conserva los datos incluso sin energía. |\n| **Propósito** | Guarda temporalmente los datos y programas que la CPU necesita usar inmediatamente. | Guarda permanentemente el sistema operativo, programas, archivos de usuario y datos. |\n| **Velocidad** | Extremadamente rápido (microsegundos). | Más lento, aunque un SSD ya es mucho más rápido que un HDD tradicional (milisegundos). |\n| **Acceso** | Acceso aleatorio y casi instantáneo a cualquier celda. | Acceso aleatorio, pero con latencia mayor y, en HDD, movimiento mecánico. |\n| **Capa

### **CUÁNTOS TOKENS CONSUMIÓ EL PROMPT**

La respuesta de la API trae un resumen (`usage`) con el número de tokens de entrada y de salida. Es la forma más directa de responder: ¿cuántos tokens consumió mi prompt?

In [6]:
# ============================================================
# PASO 5. CONSULTAR EL CONSUMO DE TOKENS
# ============================================================
#
# La propiedad response.usage contiene métricas de la llamada.
#
# prompt_tokens:
#   Tokens utilizados por nuestra entrada.
#
# completion_tokens:
#   Tokens generados en la respuesta.
#
# total_tokens:
#   Suma de entrada + salida.
#
# IMPORTANTE:
# Los valores pueden cambiar entre ejecuciones; no deben
# copiarse como números fijos.

print("Tokens del prompt:", response.usage.prompt_tokens)
print("Tokens de la respuesta:", response.usage.completion_tokens)
print("Tokens totales:", response.usage.total_tokens)


Tokens del prompt: 86
Tokens de la respuesta: 959
Tokens totales: 1045


### **MIDIENDO LA LATENCIA DE INFERENCIA**

Elegir el tamaño correcto de modelo es una decisión de ingeniería, no solo de potencia bruta. Vamos a medir cuánto tarda Llama en responder el mismo prompt.

In [7]:
# ============================================================
# PASO 6. MEDIR LA LATENCIA DEL MODELO
# ============================================================
#
# Latencia = cuánto tiempo tarda en completarse la solicitud.
#
# Guardamos el tiempo justo antes de llamar al modelo y
# volvemos a medirlo cuando termina. La diferencia es la duración.

import time

inicio = time.time()

response_tiempo = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

duracion = time.time() - inicio

print(f"Tiempo de respuesta: {duracion:.2f} segundos")

# NOTA:
# Esta segunda llamada es independiente de la primera.
# El tiempo puede variar por carga del servicio, red y longitud
# de la respuesta generada.


Tiempo de respuesta: 1.17 segundos


### **COMPARANDO DOS TAMAÑOS DE MODELO**

Repite la misma llamada usando la versión más grande de Llama disponible en Groq y compara tiempo de respuesta y calidad contra el modelo ligero.

In [8]:
# ============================================================
# PASO 7. COMPARAR DOS MODELOS LLAMA
# ============================================================
#
# Repetiremos exactamente el mismo prompt con un modelo más grande.
# Mantener el mismo prompt ayuda a que la comparación sea más justa.
#
# Según el notebook oficial de esta clase:
# - Modelo ligero: llama-3.1-8b-instant
# - Modelo grande: llama-3.3-70b-versatile

inicio = time.time()

response_grande = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

duracion_grande = time.time() - inicio

# Mostramos dos métricas básicas:
# 1. Tiempo de respuesta.
# 2. Tokens totales consumidos.
#
# OJO: "más grande" no significa automáticamente "mejor" para
# todos los casos. La idea es observar diferencias y discutirlas.

print("=== COMPARACIÓN ===")
print(f"Modelo ligero: {duracion:.2f} s — {response.usage.total_tokens} tokens")
print(f"Modelo grande:  {duracion_grande:.2f} s — {response_grande.usage.total_tokens} tokens")

print("\n=== RESPUESTA DEL MODELO GRANDE ===")
print(response_grande.choices[0].message.content)


=== COMPARACIÓN ===
Modelo ligero: 1.17 s — 1045 tokens
Modelo grande:  2.63 s — 1210 tokens

=== RESPUESTA DEL MODELO GRANDE ===
**RAM (Memoria de Acceso Aleatorio) vs. Almacenamiento (Disco duro/SSD)**  

| Característica | RAM | Almacenamiento |
|----------------|-----|----------------|
| **Tipo de memoria** | Volátil: pierde su contenido al apagar o reiniciar la máquina. | No volátil: conserva los datos aunque se corte la energía. |
| **Función principal** | Proveer a la CPU un espacio rápido donde leer y escribir los datos e instrucciones que está ejecutando en ese momento. | Guardar de forma permanente todos los archivos, sistemas operativos, programas, fotos, videos, etc. |
| **Velocidad** | Mucho más rápida (nanosegundos). Las lecturas/escrituras pueden ser cientos de GB/s en DDR4/DDR5. | Más lenta (micro‑ a milisegundos). Un SSD SATA llega a ~500 MB/s, un SSD NVMe a varios GB/s; un HDD mecánico está en ~100 MB/s. |
| **Capacidad típica** | 4 – 64 GB en ordenadores de escritori

---
# **EJEMPLO DIDÁCTICO DE CHALLENGE: COMPARADOR DE MODELOS LLAMA**

> **Importante:** esta sección es un **ejemplo de apoyo para explicar el Challenge al alumnado**.  
> No sustituye las instrucciones específicas que aparezcan en la sección oficial del Challenge dentro del Colab de Campus.

El Challenge descrito para esta clase busca que puedas reutilizar lo aprendido en el Hands On para:

1. Conectarte a Groq utilizando **Colab Secrets**.
2. Consultar más de un modelo.
3. Medir **tiempo de respuesta** y **tokens consumidos**.
4. Guardar cada resultado en un **diccionario**.
5. Organizar todos los diccionarios dentro de una **lista**.
6. Comparar los resultados obtenidos.

La idea central es pasar de ejecutar cada modelo manualmente a construir un pequeño comparador reproducible.


## **Antes del código: ¿qué estructura vamos a construir?**

Para cada modelo queremos guardar información parecida a esta:

```python
{
    "modelo": "nombre_del_modelo",
    "tiempo_segundos": 1.25,
    "tokens_prompt": 20,
    "tokens_respuesta": 150,
    "tokens_totales": 170,
    "respuesta": "..."
}
```

Eso es un **diccionario**: cada dato tiene un nombre (clave) y un valor.

Como tendremos un diccionario por cada modelo, los guardaremos dentro de una **lista**:

```python
resultados = [
    {...resultado del modelo 1...},
    {...resultado del modelo 2...}
]
```


In [9]:
# ============================================================
# CHALLENGE - PASO 1. DEFINIR LOS MODELOS A COMPARAR
# ============================================================
#
# Una lista permite guardar varios elementos en orden.
# Aquí almacenamos los nombres de los modelos que probaremos.
#
# Usamos los mismos nombres presentes en el Hands On proporcionado.

modelos = [
    "openai/gpt-oss-20b",
    "openai/gpt-oss-120b"
]

# Usaremos el mismo prompt para ambos modelos.
prompt_challenge = "Explica de forma sencilla qué es un modelo de lenguaje grande (LLM)."

# En esta lista iremos guardando los resultados de cada ejecución.
resultados = []

print("Modelos a comparar:")
for modelo in modelos:
    print("-", modelo)


Modelos a comparar:
- openai/gpt-oss-20b
- openai/gpt-oss-120b


In [10]:
# ============================================================
# CHALLENGE - PASO 2. CONSULTAR CADA MODELO Y GUARDAR MÉTRICAS
# ============================================================
#
# El ciclo for recorre la lista "modelos".
# En cada vuelta:
# 1. Guardamos el tiempo inicial.
# 2. Consultamos el modelo actual.
# 3. Calculamos la duración.
# 4. Extraemos los tokens.
# 5. Creamos un diccionario.
# 6. Añadimos ese diccionario a la lista resultados.

import time

for modelo in modelos:

    # 1) Iniciar cronómetro
    inicio = time.time()

    # 2) Hacer la llamada a Groq
    respuesta = client.chat.completions.create(
        model=modelo,
        messages=[
            {
                "role": "user",
                "content": prompt_challenge
            }
        ]
    )

    # 3) Calcular la duración
    tiempo_segundos = time.time() - inicio

    # 4) Crear un diccionario con las métricas de esta ejecución
    resultado = {
        "modelo": modelo,
        "tiempo_segundos": tiempo_segundos,
        "tokens_prompt": respuesta.usage.prompt_tokens,
        "tokens_respuesta": respuesta.usage.completion_tokens,
        "tokens_totales": respuesta.usage.total_tokens,
        "respuesta": respuesta.choices[0].message.content
    }

    # 5) Añadir el diccionario a la lista general
    resultados.append(resultado)

print("Comparación terminada.")


Comparación terminada.


In [11]:
# ============================================================
# CHALLENGE - PASO 3. MOSTRAR LOS RESULTADOS DE FORMA ORDENADA
# ============================================================
#
# Ahora recorremos la lista de resultados.
# Cada elemento de la lista es un diccionario.
#
# Podemos acceder a cada dato usando su clave:
# resultado["modelo"]
# resultado["tokens_totales"]
# etc.

for resultado in resultados:
    print("=" * 70)
    print("Modelo:", resultado["modelo"])
    print(f"Tiempo: {resultado['tiempo_segundos']:.2f} segundos")
    print("Tokens del prompt:", resultado["tokens_prompt"])
    print("Tokens de respuesta:", resultado["tokens_respuesta"])
    print("Tokens totales:", resultado["tokens_totales"])
    print("\nRespuesta:")
    print(resultado["respuesta"])
    print()


Modelo: openai/gpt-oss-20b
Tiempo: 0.64 segundos
Tokens del prompt: 87
Tokens de respuesta: 365
Tokens totales: 452

Respuesta:
Un **modelo de lenguaje grande** (LLM, por sus siglas en inglés) es básicamente un programa de computadora que ha “leído” enormes cantidades de texto (libros, artículos, páginas web, etc.) y ha aprendido a predecir cuál palabra o frase viene después de las que ya están escritas.

Imagina que le das a un niño un libro y cada vez que le preguntas “¿qué sigue?”, él intenta adivinar la próxima palabra basándose en todo lo que ya ha leído. Un LLM funciona de forma muy parecida, pero con la ayuda de una máquina: procesa miles de millones de ejemplos y ajusta internamente millones de pequeñas piezas de cálculo (los parámetros) hasta que puede generar textos coherentes, responder preguntas, resumir información, escribir historias, entre otras tareas.

**Puntos clave:**

1. **Entrenamiento masivo** – Se alimenta con textos gigantes (billones de palabras) para aprender 

## **¿Cómo interpretar el resultado?**

No busques únicamente cuál modelo tuvo el número más pequeño.

Compara al menos:

- **Tiempo de respuesta:** ¿cuál respondió más rápido en esta ejecución?
- **Tokens totales:** ¿cuánto contenido procesó y generó cada uno?
- **Respuesta generada:** ¿observas diferencias en extensión, claridad o detalle?

Recuerda: una única ejecución **no demuestra de forma definitiva** que un modelo siempre será más rápido o mejor. Este ejemplo sirve para practicar la medición y organización de resultados que solicita el Challenge.


In [12]:
# ============================================================
# CHALLENGE - PASO 4. EJEMPLO DE COMPARACIÓN AUTOMÁTICA
# ============================================================
#
# min(..., key=...) permite encontrar el diccionario que tenga
# el valor más pequeño en una métrica determinada.
#
# Aquí lo usamos SOLO para identificar:
# - menor tiempo medido en esta ejecución;
# - menor cantidad de tokens totales en esta ejecución.
#
# No significa que ese modelo sea "el mejor" en general.

mas_rapido = min(resultados, key=lambda x: x["tiempo_segundos"])
menos_tokens = min(resultados, key=lambda x: x["tokens_totales"])

print("En esta ejecución:")
print("- Menor tiempo:", mas_rapido["modelo"])
print("- Menor cantidad de tokens:", menos_tokens["modelo"])


En esta ejecución:
- Menor tiempo: openai/gpt-oss-20b
- Menor cantidad de tokens: openai/gpt-oss-20b


## **Mini ejercicio para explicarlo en clase**

Después de ejecutar el ejemplo, pide al alumnado modificar **solo una cosa**:

1. Cambiar `prompt_challenge` por una pregunta propia.
2. Ejecutar nuevamente la comparación.
3. Observar si cambian los tiempos y los tokens.
4. Explicar en palabras qué hace:
   - la lista `modelos`,
   - el diccionario `resultado`,
   - la lista `resultados`,
   - y el ciclo `for`.

### Preguntas de comprobación

- ¿Por qué utilizamos el mismo prompt con ambos modelos?
- ¿Dónde se obtiene el número de tokens?
- ¿Qué diferencia hay entre una lista y un diccionario?
- ¿Por qué el tiempo puede variar entre ejecuciones?
- ¿Por qué no debemos concluir que un modelo es siempre mejor solo con una prueba?

### Relación con el Challenge oficial

Este ejemplo demuestra las tres competencias descritas en el Challenge que compartió el curso: **integración con Groq mediante Colab Secrets, medición de desempeño y organización de resultados con listas y diccionarios**. Para la entrega real, el alumno debe seguir las instrucciones exactas que aparezcan en la sección oficial del Challenge.
